# Chuẩn bị Cityscapes trên Kaggle

Upload notebook này vào Kaggle. Thêm dataset riêng của bạn chứa Cityscapes và `project-code.zip` đi kèm. Sửa hai đường dẫn trong cell đầu. Không public ảnh/nhãn/raw data. Notebook chưa chạy trên Kaggle; code đã được kiểm chứng local. Chỉ dùng train/val Cityscapes, không mở ACDC. Cell chuyển nhãn có thể chạy vài phút.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, json, hashlib
# EDIT these two paths to match your private Kaggle input datasets.
SOURCE_ZIP = Path('/kaggle/input/YOUR_CODE_DATASET/project-code.zip')
CITYSCAPES_ROOT = Path('/kaggle/input/YOUR_CITYSCAPES_DATASET/cityscapes')
REPO = Path('/kaggle/working/adverse-weather-segmentation')
# Kaggle may automatically extract an uploaded ZIP into the dataset root.
SOURCE_ROOT = None  # Optional folder containing src/, scripts/, configs/ if ZIP was extracted
assert (CITYSCAPES_ROOT/'leftImg8bit/train').is_dir(), 'Set CITYSCAPES_ROOT to the folder containing leftImg8bit and gtFine'
REPO.mkdir(parents=True,exist_ok=True)
if SOURCE_ZIP.is_file():
    with zipfile.ZipFile(SOURCE_ZIP) as bundle:
        for member in bundle.infolist():
            target=(REPO/member.filename).resolve()
            assert target.is_relative_to(REPO.resolve()), 'Unsafe archive member'
        bundle.extractall(REPO)
    print('Code ZIP SHA256:',hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest())
else:
    import shutil
    source=Path(SOURCE_ROOT) if SOURCE_ROOT else SOURCE_ZIP.parent
    assert all((source/name).is_dir() for name in ['src','scripts','configs']), f'Set SOURCE_ROOT to extracted code folder: {source}'
    for name in ['src','scripts','configs','tests','requirements.txt','pyproject.toml']:
        item=source/name
        if item.is_dir():
            shutil.copytree(item,REPO/name,dirs_exist_ok=True)
        elif item.is_file():
            shutil.copy2(item,REPO/name)
    print('Copied extracted source from:',source)
os.chdir(REPO)
sys.path.insert(0,str(REPO))
os.environ['OMP_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],check=True)
print('Working directory:',REPO)


In [ ]:
# Enable Internet for first-time package and pretrained-weight downloads.
# Preserve Kaggle's CUDA PyTorch installation.
run('-m','pip','install','transformers==4.57.6','numpy>=1.26,<3','Pillow>=10,<12','PyYAML>=6,<7','tqdm>=4.66,<5','pytest>=8,<10','matplotlib>=3.8,<4')
import torch, transformers, yaml
print('torch',torch.__version__,'transformers',transformers.__version__)
print('CUDA available:',torch.cuda.is_available())


In [ ]:
# Reuse valid processed masks only when you deliberately set this path.
PROCESSED_ROOT = REPO/'data/processed/cityscapes'
if PROCESSED_ROOT.exists():
    raise RuntimeError('Processed folder already exists. Validate/reuse it deliberately or use a NEW writable folder; no silent overwrite.')
run('scripts/convert_labels.py','--root',CITYSCAPES_ROOT,'--output-root',PROCESSED_ROOT,'--workers','4')
run('scripts/make_manifest.py','--dataset','cityscapes','--root',CITYSCAPES_ROOT,'--mask-root',PROCESSED_ROOT)
run('scripts/validate_manifest.py','--manifest','data/manifests/cityscapes_train.csv','data/manifests/cityscapes_val.csv','--check-files','--check-labels')


In [ ]:
run('scripts/audit_labels.py','--manifest','data/manifests/cityscapes_train.csv','data/manifests/cityscapes_val.csv','--check-files','--check-labels','--raw-root',CITYSCAPES_ROOT,'--out','results/kaggle-cityscapes-qa/audit.json')
print(json.loads(Path('results/kaggle-cityscapes-qa/audit.json').read_text())['status'])


## Lưu kết quả

Save Version của Kaggle với outputs để giữ processed masks và báo cáo. Notebook 02 có thể nhận thư mục `data/processed/cityscapes` này qua input `PREPARED_ROOT`, hoặc tự chuyển lại nhãn trong phiên mới. CSV phải được tạo lại khi mount path thay đổi. Không dùng kết quả chuẩn bị như số liệu segmentation.
